# 🎬 Estudio de Video Continuo: Flux + Wan 2.1 (versión corregida)
Requiere GPU con ≥16 GB (T4 funciona con Wan 14B fp8 pero lento; L4/A100 recomendado).

In [ ]:
# PASO 1: GPU
import torch
print(f"✅ GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "❌ No hay GPU. Entorno de ejecución > Cambiar tipo > GPU")

In [ ]:
# PASO 2: Instalación
!pip install gradio opencv-python Pillow --quiet
!git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI
!pip install -r requirements.txt --quiet
print("✅ Instalación completada.")

In [ ]:
# PASO 3: Modelos
import os
!apt-get install -y aria2 -qq
M = '/content/ComfyUI/models'
for d in ['checkpoints','diffusion_models','vae','text_encoders','clip_vision']:
    os.makedirs(f'{M}/{d}', exist_ok=True)
HF = 'https://huggingface.co'
WAN = f'{HF}/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files'
def dl(url, folder, name):
    !aria2c -c -x 16 -s 16 -k 1M "{url}" -d {M}/{folder} -o {name}
# Flux dev fp8: checkpoint completo (incluye CLIP, T5 y VAE)
dl(f'{HF}/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors', 'checkpoints', 'flux1-dev-fp8.safetensors')
# Wan 2.1 imagen->video 480p (el 1.3B NO existe para i2v; solo 14B)
dl(f'{WAN}/diffusion_models/wan2.1_i2v_480p_14B_fp8_scaled.safetensors', 'diffusion_models', 'wan2.1_i2v_480p_14B_fp8_scaled.safetensors')
dl(f'{WAN}/vae/wan_2.1_vae.safetensors', 'vae', 'wan_2.1_vae.safetensors')
dl(f'{WAN}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'text_encoders', 'umt5_xxl_fp8_e4m3fn_scaled.safetensors')
dl(f'{WAN}/clip_vision/clip_vision_h.safetensors', 'clip_vision', 'clip_vision_h.safetensors')
print("✅ Descarga finalizada.")

In [ ]:
# PASO 4: Servidor ComfyUI
import subprocess, time, urllib.request, os
os.chdir('/content/ComfyUI')
srv = subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188'])
for _ in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats'); print("✅ Servidor activo."); break
    except Exception: time.sleep(2)
else: print("❌ El servidor no arrancó; revisa los logs.")

In [ ]:
# PASO 5: Interfaz
import gradio as gr, cv2, json, urllib.request, urllib.error, time, os
from PIL import Image

HOST = '127.0.0.1:8188'
OUT = '/content/ComfyUI/output'
INP = '/content/ComfyUI/input'
os.makedirs(OUT, exist_ok=True); os.makedirs(INP, exist_ok=True)

def run_workflow(wf, poll=3, timeout=3600):
    """Envía el workflow, espera y devuelve (outputs, error). Muestra el error real de ComfyUI."""
    req = urllib.request.Request(f'http://{HOST}/prompt', data=json.dumps({'prompt': wf}).encode(),
                                 headers={'Content-Type': 'application/json'})
    try:
        pid = json.loads(urllib.request.urlopen(req).read())['prompt_id']
    except urllib.error.HTTPError as e:
        return None, f"ComfyUI rechazó el workflow: {e.read().decode()[:1500]}"
    except Exception as e:
        return None, f'No se pudo contactar con ComfyUI: {e}'
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            h = json.loads(urllib.request.urlopen(f'http://{HOST}/history/{pid}').read())
            if pid in h:
                st = h[pid].get('status', {})
                if st.get('status_str') == 'error':
                    msgs = [m for m in st.get('messages', []) if m[0] == 'execution_error']
                    return None, f"Error de ejecución: {msgs[-1][1].get('exception_message') if msgs else st}"
                return h[pid]['outputs'], None
        except Exception: pass
        time.sleep(poll)
    return None, 'Tiempo agotado.'

def primer_archivo(outputs, clave):
    for o in outputs.values():
        for f in o.get(clave, []):
            return os.path.join(OUT, f.get('subfolder', ''), f['filename'])
    return None

def extraer_ultimo_fotograma(video_path):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    img = None
    if total > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, total - 1)
        ok, fr = cap.read()
        if ok: img = Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    cap.release()
    if img is None: return None, None
    path = f'/content/ultimo_frame_{int(time.time())}.png'
    img.save(path)
    return img, path

def generar_imagen_maestra(prompt, neg, ancho, alto):
    wf = {
      '1': {'class_type':'CheckpointLoaderSimple','inputs':{'ckpt_name':'flux1-dev-fp8.safetensors'}},
      '4': {'class_type':'CLIPTextEncode','inputs':{'text':prompt,'clip':['1',1]}},
      '5': {'class_type':'CLIPTextEncode','inputs':{'text':neg or '','clip':['1',1]}},
      '10':{'class_type':'FluxGuidance','inputs':{'conditioning':['4',0],'guidance':3.5}},
      '6': {'class_type':'EmptyLatentImage','inputs':{'width':int(ancho),'height':int(alto),'batch_size':1}},
      '7': {'class_type':'KSampler','inputs':{'seed':int(time.time()),'steps':20,'cfg':1.0,'sampler_name':'euler',
            'scheduler':'simple','denoise':1.0,'model':['1',0],'positive':['10',0],'negative':['5',0],'latent_image':['6',0]}},
      '8': {'class_type':'VAEDecode','inputs':{'samples':['7',0],'vae':['1',2]}},
      '9': {'class_type':'SaveImage','inputs':{'filename_prefix':'Master_Image','images':['8',0]}},
    }
    outs, err = run_workflow(wf, poll=2)
    if err: return None, f'❌ {err}'
    p = primer_archivo(outs, 'images')
    if p and os.path.exists(p): return Image.open(p).convert('RGB'), '✅ Imagen generada.'
    return None, '❌ ComfyUI terminó pero no devolvió imagen.'

def generar_clip_i2v(prompt, neg, dur, imagen_path, n):
    if not imagen_path or not os.path.exists(imagen_path):
        return None, '❌ Imagen inicial inválida.', None, None
    nombre = f'input_clip_{n}_{int(time.time())}.png'
    img = Image.open(imagen_path).convert('RGB'); img.save(f'{INP}/{nombre}')
    w, h = (img.width // 16) * 16, (img.height // 16) * 16
    w, h = min(w, 832), min(h, 832)
    length = int(dur) * 16 + 1   # Wan usa frames = 4k+1 a 16 fps
    wf = {
      '1': {'class_type':'UNETLoader','inputs':{'unet_name':'wan2.1_i2v_480p_14B_fp8_scaled.safetensors','weight_dtype':'default'}},
      '2': {'class_type':'VAELoader','inputs':{'vae_name':'wan_2.1_vae.safetensors'}},
      '3': {'class_type':'CLIPLoader','inputs':{'clip_name':'umt5_xxl_fp8_e4m3fn_scaled.safetensors','type':'wan','device':'default'}},
      '11':{'class_type':'CLIPVisionLoader','inputs':{'clip_name':'clip_vision_h.safetensors'}},
      '4': {'class_type':'LoadImage','inputs':{'image':nombre}},
      '12':{'class_type':'CLIPVisionEncode','inputs':{'clip_vision':['11',0],'image':['4',0],'crop':'none'}},
      '5': {'class_type':'CLIPTextEncode','inputs':{'text':prompt,'clip':['3',0]}},
      '6': {'class_type':'CLIPTextEncode','inputs':{'text':neg or '','clip':['3',0]}},
      '13':{'class_type':'ModelSamplingSD3','inputs':{'model':['1',0],'shift':8.0}},
      '14':{'class_type':'WanImageToVideo','inputs':{'positive':['5',0],'negative':['6',0],'vae':['2',0],'width':w,'height':h,
            'length':length,'batch_size':1,'clip_vision_output':['12',0],'start_image':['4',0]}},
      '7': {'class_type':'KSampler','inputs':{'seed':int(time.time()),'steps':20,'cfg':6.0,'sampler_name':'uni_pc',
            'scheduler':'simple','denoise':1.0,'model':['13',0],'positive':['14',0],'negative':['14',1],'latent_image':['14',2]}},
      '8': {'class_type':'VAEDecode','inputs':{'samples':['7',0],'vae':['2',0]}},
      '15':{'class_type':'CreateVideo','inputs':{'images':['8',0],'fps':16.0}},
      '9': {'class_type':'SaveVideo','inputs':{'video':['15',0],'filename_prefix':f'video/Proyecto_Clip_{n}','format':'mp4','codec':'h264'}},
    }
    outs, err = run_workflow(wf, poll=4)
    if err: return None, f'❌ {err}', None, None
    vp = primer_archivo(outs, 'images') or primer_archivo(outs, 'gifs')
    if not vp or not os.path.exists(vp): return None, '❌ No se encontró el video generado.', None, None
    pil, p = extraer_ultimo_fotograma(vp)
    return vp, f'✅ Clip #{n} renderizado.', pil, p

with gr.Blocks(title='Estudio Video Wan 2.1') as demo:
    gr.Markdown('# 🎬 Estudio de Video Continuo')
    estado_clip = gr.State(1); estado_path = gr.State(None)
    with gr.Tabs() as tabs:
        with gr.TabItem('1️⃣ Imagen Maestra', id=0):
            with gr.Row():
                with gr.Column():
                    t2i_prompt = gr.Textbox(label='Prompt', lines=3)
                    t2i_neg = gr.Textbox(label='Prompt Negativo', value='blurry, ugly, distorted')
                    t2i_ancho = gr.Slider(512, 1280, value=832, step=64, label='Ancho')
                    t2i_alto = gr.Slider(512, 1280, value=480, step=64, label='Alto')
                    btn_m = gr.Button('🎨 Generar Imagen', variant='primary')
                with gr.Column():
                    out_master = gr.Image(label='Resultado', type='pil')
                    st_m = gr.Textbox(label='Estado')
                    btn_t = gr.Button('➡️ Usar en Video')
        with gr.TabItem('2️⃣ Secuenciador de Video', id=1):
            lbl = gr.Markdown('### 📌 Configurando: Clip #1')
            with gr.Row():
                with gr.Column():
                    v_prompt = gr.Textbox(label='Acción para el Video', lines=3)
                    v_neg = gr.Textbox(label='Prompt Negativo', value='blurry, static, distorted')
                    v_dur = gr.Slider(2, 5, value=3, step=1, label='Duración (s)')
                    v_frame = gr.Image(label='Fotograma Inicial', type='pil')
                    btn_c = gr.Button('🚀 Generar Clip', variant='primary')
                with gr.Column():
                    out_vid = gr.Video(label='Video'); st_v = gr.Textbox(label='Estado')
                    out_last = gr.Image(label='Último fotograma (siguiente inicio)', type='pil')

    btn_m.click(generar_imagen_maestra, [t2i_prompt, t2i_neg, t2i_ancho, t2i_alto], [out_master, st_m])

    def fn_trans(img):
        if img is None: return gr.update(), None, gr.update(), '❌ Falta imagen.'
        p = f'/content/master_{int(time.time())}.png'; img.save(p)
        return img, p, gr.Tabs(selected=1), '✅ Enviada al secuenciador.'
    btn_t.click(fn_trans, [out_master], [v_frame, estado_path, tabs, st_m])

    def fn_clip(p, n, d, f_pil, f_path, c):
        if f_pil is not None:   # usa siempre lo que se ve en pantalla (permite edición manual)
            f_path = f'/content/manual_{int(time.time())}.png'; f_pil.save(f_path)
        v, s, npil, npath = generar_clip_i2v(p, n, d, f_path, c)
        if v is None: return None, s, None, gr.update(), f_path, c, gr.update()
        return v, s, npil, npil, npath, c + 1, f'### 📌 Configurando: Clip #{c + 1}'
    btn_c.click(fn_clip, [v_prompt, v_neg, v_dur, v_frame, estado_path, estado_clip],
                [out_vid, st_v, out_last, v_frame, estado_path, estado_clip, lbl])

demo.launch(inline=True, debug=False)